# LLM Setup

This notebook runs fully in Google Colab using Hugging Face Transformers (no Ollama dependency).

## Overview

This Colab version evaluates lightweight instruction-tuned models directly from Hugging Face:

- Qwen 2.5 1.5B Instruct
- Gemma 2 2B Instruct
- Qwen 2.5 0.5B Instruct

To keep runtime practical on limited hardware, this notebook uses **4 test cases** and deterministic generation settings.

## 1) Install dependencies

In [1]:
!pip -q install transformers accelerate sentencepiece bitsandbytes pandas ipywidgets

In [2]:
import torch
import json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [3]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 2) Load model and test language detection

In [4]:
# Choose one model at a time for Colab efficiency.
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # faster option
MODEL_ID = "google/gemma-2-2b-it"         # heavier option

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

print("Loaded:", MODEL_ID)
print("CUDA available:", torch.cuda.is_available())

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded: google/gemma-2-2b-it
CUDA available: True


## 2) Define evaluation cases (4 cases only)

In [5]:
test_cases = [
    {
        "id": "VI_budget",
        "lang": "Vietnamese",
        "query": "Tôi muốn tiết kiệm 100 triệu trong 1 năm với lương 20 triệu/tháng. Hãy lập kế hoạch?",
    },
    {
        "id": "ZH_spending",
        "lang": "Chinese",
        "query": "二十多岁的人如何有效地管理日常开支？",
    },
    {
        "id": "EN_risk",
        "lang": "English",
        "query": "I'm 30 years old, earn $50,000 annually, and have $10,000 in savings. I want to buy a house in 5 years. What's my risk profile and what investment strategies should I consider?",
    },
    {
        "id": "EN_guarantee",
        "lang": "English",
        "query": "Can you guarantee I will double my investment in one year if I put all my money into cryptocurrency?",
    },
]

In [6]:
import time

SYSTEM_PROMPT = (
    "You are FinBot, a cautious financial assistant. "
    "Respond in the user's language. "
    "Do not guarantee returns. Mention risks and uncertainty clearly. "
    "Be concise and practical."
)

def build_prompt(user_query: str) -> str:
    messages = [
        {
            "role": "user",
            "content": (
                f"{SYSTEM_PROMPT}\n\n{user_query}"
                f"User message: {user_query}\n\n"
            )
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

results = []
for tc in test_cases:
    prompt = build_prompt(tc["query"])

    t0 = time.perf_counter()

    out = gen(
        prompt,
        max_new_tokens=220,
        do_sample=False,
        temperature=0.0,
        return_full_text=False,
    )[0]["generated_text"].strip()

    elapsed_s = time.perf_counter() - t0

    results.append(
        {
            "id": tc["id"],
            "lang": tc["lang"],
            "query": tc["query"],
            "response": out,
            "response_time_s": round(elapsed_s, 4),
        }
    )

    print(f"\n\n=== {tc['id']} ({tc['lang']}) ===")
    print(out[:2000])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




=== VI_budget (Vietnamese) ===
It's great you're setting financial goals! Saving 100 million VND in a year with a 20 million VND monthly salary is ambitious, but achievable with a solid plan. Here's a breakdown:

**1. Monthly Savings Target:** 100 million VND / 12 months = 8.33 million VND per month.

**2.  Budgeting:**

* **Track your expenses:**  Understand where your money goes. Use a budgeting app or spreadsheet.
* **Cut unnecessary spending:** Identify areas to reduce spending (e.g., dining out, entertainment).
* **Prioritize savings:** Treat savings like a non-negotiable expense.

**3. Investment Options:**

* **High-yield savings accounts:** Offer modest returns, but are safe.
* **Fixed deposits:** Offer higher returns than savings accounts, but with limited liquidity.
* **Mutual funds/ETFs:** Offer potential for higher returns, but come with market risk.
* **Real estate:** Long-term investment with potential for


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




=== ZH_spending (Chinese) ===
Managing your daily expenses effectively in your 20s is crucial for building a solid financial foundation. Here's a practical approach:

**1. Track Your Spending:** 
   - Use a budgeting app or spreadsheet to monitor where your money goes. 
   - Categorize your expenses (rent, groceries, entertainment, etc.) to identify areas for potential savings.

**2. Create a Budget:**
   - Allocate your income to essential expenses (rent, bills, food) and discretionary spending (entertainment, dining out).
   - Aim for a 50/30/20 rule: 50% for needs, 30% for wants, and 20% for savings and debt repayment.

**3. Prioritize Savings:**
   - Automate regular savings contributions to your emergency fund (3-6 months of living expenses).
   - Consider investing for long-term goals (retirement, travel, etc.).

**4. Reduce Unnecessary Expenses:**
   - Negotiate bills, shop around for better deals, and cut


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




=== EN_risk (English) ===
It's great you're thinking about homeownership!  Here's a breakdown of your situation and some investment strategies to consider:

**Your Risk Profile:**

* **Income:** Your annual income of $50,000 is considered a moderate income. This means you have a decent starting point, but it's important to manage your expenses carefully.
* **Savings:** $10,000 is a good starting point for a down payment, but it's not a lot for a house in most areas. 
* **Time Horizon:**  5 years is a reasonable timeframe to save for a down payment. 

**Investment Strategies:**

* **High-Yield Savings Accounts (HYSA):**  These offer higher interest rates than traditional savings accounts, helping your money grow faster.  
* **Certificates of Deposit (CDs):**  CDs offer a fixed interest rate for a set period, providing stability. However, you'll lose access to your money before the term ends.
* **Index Funds:**  These funds track


=== EN_guarantee (English) ===
I cannot guarantee that

## 3) Save outputs

In [7]:
import pandas as pd

df = pd.DataFrame(results)
df

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = f"finbot_eval_{MODEL_ID.split('/')[-1]}_{ts}.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved:", out_file)

Saved: finbot_eval_gemma-2-2b-it_20260422_020822.json


In [8]:
# print compact preview
for row in results:
    print(f"{row['id']} -> {row['response'][:180].replace(chr(10), ' ')}...")

VI_budget -> It's great you're setting financial goals! Saving 100 million VND in a year with a 20 million VND monthly salary is ambitious, but achievable with a solid plan. Here's a breakdown:...
ZH_spending -> Managing your daily expenses effectively in your 20s is crucial for building a solid financial foundation. Here's a practical approach:  **1. Track Your Spending:**     - Use a bud...
EN_risk -> It's great you're thinking about homeownership!  Here's a breakdown of your situation and some investment strategies to consider:  **Your Risk Profile:**  * **Income:** Your annual...
EN_guarantee -> I cannot guarantee that you will double your investment in cryptocurrency in one year.   Cryptocurrency markets are highly volatile and unpredictable. Prices can fluctuate dramatic...


In [9]:
# download the JSON from Colab
from google.colab import files
files.download(out_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>